Evolve the table's schema two ways: append a new column using mergeSchema, then change an
existing column's type using overwriteSchema; document the difference in what each requires.

In [0]:
%python
from pyspark.sql.functions import *

# adding the colummmn order_status
new_df = spark.table("cyntexa_dev.bronze.orders_bronze").withColumn("order_status", lit("pending"))

new_df.write.format("delta").mode("append").option("mergeSchema" ,"true").saveAsTable("cyntexa_dev.bronze.orders_bronze")

In [0]:
%python
updated_df = (
    spark.table("cyntexa_dev.bronze.orders_bronze")
    .withColumn(
        "order_date",
        try_to_date(col("order_date"), "yyyy/MM/dd")
    )
)

(
    updated_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cyntexa_dev.bronze.orders_bronze")
)


Set up an Autoloader stream ingesting from a folder, then drop 2 more files into the folder and
confirm they're picked up automatically.

In [0]:
%python
# Set up Auto Loader stream to ingest CSV files
from pyspark.sql.functions import *

# Define paths
source_path = "/Volumes/dev/autoloader/raw/sales/"
schema_location = "/Volumes/dev/autoloader/raw/autoloader_schemaLocation/"
checkpoint_path = "/Volumes/dev/autoloader/raw/autoloader_checkpoint/"
target_table = "dev.autoloader.sales_autoloader"

# Read stream using Auto Loader
df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("header", "true")
    .option("inferSchema", "true")
    .load(source_path)
)

# Write stream to Delta table
query = (df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)  # Process all available files then stop
    .toTable(target_table)
)

# print(f"Auto Loader stream started. Monitoring folder: {source_path}")
# print(f"Target table: {target_table}")


6. Use RESTORE to roll a table back to a version before a bad schema change, and describe what
happens to the versions that were created after the point you restored to.

-->

Added a dummy column to the table to change the schema. After validating the schema change, the table was restored to its previous version using Delta Lake Time Travel.

In [0]:
describe history dev.autoloader.sales_autoloader;

In [0]:

ALTER TABLE dev.autoloader.sales_autoloader
ADD COLUMN (dummy_col string);


In [0]:
RESTORE dev.autoloader.sales_autoloader TO VERSION AS OF 3;

In [0]:
DESCRIBE dev.autoloader.sales_autoloader;